# Web scraping Les Echos avec Selenium

## Objectif

Rechercher les articles correspondant au terme **ia**, ouvrir chaque article et récupérer :

- son titre ;
- son petit résumé lorsqu'il existe ;
- son URL.


# Étape 1 - Installer les bibliothèques


In [31]:
# %pip install selenium pandas

# Étape 2 - Importer les bibliothèques


In [32]:
from pathlib import Path

import pandas as pd
from selenium import webdriver
from selenium.common.exceptions import  TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

TIMEOUT = 15
MAX_ARTICLES = 20  
NOMBRE_PAGES = 1   

# Étape 3 - Préparer les fonctions utiles

## Fonctions de recherche d'éléments


In [33]:
def trouver_premier(driver, selecteurs):
    """Renvoie le premier élément trouvé parmi plusieurs sélecteurs."""
    for by, valeur in selecteurs:
        elements = driver.find_elements(by, valeur)
        if elements:
            return elements[0]
    return None


def texte_premier(driver, selecteurs, valeur_par_defaut=""):
    """Renvoie le texte du premier élément trouvé."""
    element = trouver_premier(driver, selecteurs)
    return element.text.strip() if element else valeur_par_defaut


def attribut_premier(driver, selecteurs, attribut, valeur_par_defaut=""):
    """Renvoie un attribut du premier élément trouvé."""
    element = trouver_premier(driver, selecteurs)
    if not element:
        return valeur_par_defaut
    return (element.get_attribute(attribut) or valeur_par_defaut).strip()

# Étape 4 - Ouvrir Chrome et accéder au site

## Configuration du navigateur



In [34]:
options = webdriver.ChromeOptions()
options.binary_location = r"C:\Program Files\Google\Chrome\Application\chrome.exe"
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, TIMEOUT)

driver.get("https://www.lesechos.fr/")

if driver.title == "Access Denied":
    driver.quit()
    raise RuntimeError("Les Echos a refusé cette session Chrome. Relancer le navigateur sans mode headless.")

print("Page ouverte :", driver.title)

Page ouverte : Les Echos : actualités en direct, Économie, Finance, Marchés, Politique, Entreprises, Start-up - Les Echos


# Étape 5 - Accepter les cookies




In [35]:
try:
    bouton_accepter = WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.XPATH, "//button[normalize-space()='Accepter']"))
    )
    bouton_accepter.click()
    print("Cookies acceptés.")
except TimeoutException:
    print("Le bandeau de cookies n'est pas affiché ou le choix a déjà été enregistré.")

Cookies acceptés.


# Étape 6 - Rechercher les articles sur l'IA

## Ouvrir la recherche


In [36]:
bouton_recherche = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "button[aria-label='Recherche']"))
)
bouton_recherche.click()

champ_recherche = wait.until(
    EC.visibility_of_element_located((By.CSS_SELECTOR, "input[name='query']"))
)
champ_recherche.clear()
champ_recherche.send_keys("ia")
champ_recherche.send_keys(Keys.ENTER)

wait.until(EC.url_contains("/recherche?search=ia"))
print("Page de résultats :", driver.current_url)

Page de résultats : https://www.lesechos.fr/recherche?search=ia&searchType=posts


# Étape 7 - Récupérer les liens des articles

## Parcourir les pages de résultats


In [37]:
def recuperer_liens_articles(driver, nombre_pages=1, maximum=None):
    liens = []
    liens_deja_vus = set()

    for page in range(1, nombre_pages + 1):
        url_page = "https://www.lesechos.fr/recherche?search=ia&searchType=posts"
        if page > 1:
            url_page += f"&page={page}"

        driver.get(url_page)
        WebDriverWait(driver, TIMEOUT).until(
            EC.presence_of_all_elements_located((By.XPATH, "//a[.//h3]"))
        )

        for carte in driver.find_elements(By.XPATH, "//a[.//h3]"):
            url = (carte.get_attribute("href") or "").split("?")[0]
            if url and url not in liens_deja_vus:
                liens.append(url)
                liens_deja_vus.add(url)

            if maximum and len(liens) >= maximum:
                return liens

    return liens


liens_articles = recuperer_liens_articles(driver, NOMBRE_PAGES, MAX_ARTICLES)
print(f"{len(liens_articles)} article(s) collecté(s).")
liens_articles[:5]

20 article(s) collecté(s).


['https://www.lesechos.fr/pme-regions/actualite-pme/on-a-mis-les-ingredients-mais-maintenant-il-faut-que-la-mayonnaise-prenne-comment-valerie-pecresse-veut-redonner-un-second-souffle-a-paris-saclay-2234760',
 'https://www.lesechos.fr/partenaires/vivatech-2026/conor-pierce-samsung-linnovation-nest-jamais-une-course-en-solitaire-aucune-entreprise-ne-peut-construire-seule-lavenir-de-lia-2234739',
 'https://www.lesechos.fr/partenaires/vivatech-2026/lutz-diederichs-bnp-paribas-allemagne-nous-adoptons-une-approche-agile-prudente-et-responsable-lia-ne-se-substitue-pas-a-lexpertise-de-nos-equipes-elle-vient-en-soutien-de-leur-dialogue-avec-les-clients-et-leur-prise-de-decision-2234737',
 'https://www.lesechos.fr/monde/enjeux-internationaux/le-conflit-au-moyen-orient-a-completement-annihile-nos-previsions-de-reprise-en-2026-2234715',
 'https://www.lesechos.fr/start-up/deals/la-licorne-aircall-fait-ses-premieres-acquisitions-dans-lia-2234713']

# Étape 8 - Extraire le titre et le résumé de chaque article

## Extraction sur la page de l'article

- Le titre est récupéré dans `<h1>`.
- Le petit résumé est récupéré dans `p[data-testid='post-lead']`.
- La métadonnée `meta[name='description']` sert de solution de secours, notamment pour certains articles premium.

In [38]:
def extraire_article(driver, url):
    driver.get(url)
    WebDriverWait(driver, TIMEOUT).until(
        EC.presence_of_element_located((By.TAG_NAME, "h1"))
    )

    titre = texte_premier(driver, [
        (By.TAG_NAME, "h1"),
        (By.CSS_SELECTOR, "meta[property='og:title']")
    ], valeur_par_defaut="Titre non trouvé")

    resume = texte_premier(driver, [
        (By.CSS_SELECTOR, "p[data-testid='post-lead']")
    ])

    if not resume:
        resume = attribut_premier(driver, [
            (By.CSS_SELECTOR, "meta[name='description']"),
            (By.CSS_SELECTOR, "meta[property='og:description']")
        ], "content", valeur_par_defaut="Résumé non trouvé")

    return {
        "titre": titre,
        "resume": resume,
        "url": url
    }


articles = []
for index, url in enumerate(liens_articles, start=1):
    try:
        article = extraire_article(driver, url)
        articles.append(article)
        print(f"[{index}/{len(liens_articles)}] {article['titre']}")
    except Exception as erreur:
        print(f"[{index}/{len(liens_articles)}] Article ignoré : {url} ({erreur})")

df_articles = pd.DataFrame(articles)
df_articles

[1/20] « On a mis les ingrédients mais, maintenant, il faut que la mayonnaise prenne » : comment Valérie Pécresse veut redonner un second souffle à Paris-Saclay
[2/20] Conor Pierce (Samsung) : « L’innovation n’est jamais une course en solitaire — aucune entreprise ne peut construire seule l’avenir de l’IA. »
[3/20] Lutz Diederichs (BNP Paribas Allemagne) : « Nous adoptons une approche agile, prudente et responsable. L'IA ne se substitue pas à l'expertise de nos équipes, elle vient en soutien de leur dialogue avec les clients et leur prise de décision. »
[4/20] « Le conflit au Moyen-Orient a complètement annihilé nos prévisions de reprise en 2026 »
[5/20] La licorne Aircall fait ses premières acquisitions dans l'IA
[6/20] Sêmeia lève 21 millions d'euros pour étendre la télésurveillance médicale
[7/20] « Comme un lapin pris dans les phares » : la stratégie cyber des grands groupes mise à mal par Claude Mythos
[8/20] « Les levées de fonds des géants de l'IA risquent de créer un effet d'év

,titre,resume,url
0,"« On a mis les ingrédients mais, maintenant, i...",La présidente de la région lle-de-France veut ...,https://www.lesechos.fr/pme-regions/actualite-...
1,Conor Pierce (Samsung) : « L’innovation n’est ...,"Dans cette vidéo, Conor Pierce, Président de S...",https://www.lesechos.fr/partenaires/vivatech-2...
2,Lutz Diederichs (BNP Paribas Allemagne) : « No...,Engagé dans l'intelligence artificielle depuis...,https://www.lesechos.fr/partenaires/vivatech-2...
3,« Le conflit au Moyen-Orient a complètement an...,"L'économiste en chef de l'OCDE, Stefano Scarpe...",https://www.lesechos.fr/monde/enjeux-internati...
4,La licorne Aircall fait ses premières acquisit...,La licorne franco-américaine a racheté coup su...,https://www.lesechos.fr/start-up/deals/la-lico...
5,Sêmeia lève 21 millions d'euros pour étendre l...,Le leader français de la télésurveillance médi...,https://www.lesechos.fr/pme-regions/innovateur...
6,« Comme un lapin pris dans les phares » : la s...,Les grands groupes français renforcent vaille ...,https://www.lesechos.fr/tech-medias/hightech/c...
7,« Les levées de fonds des géants de l'IA risqu...,"Les levées de capitaux record d'Alphabet, Spac...",https://www.lesechos.fr/finance-marches/marche...
8,Cybersécurité : Donald Trump régule l'IA « sur...,"Après des semaines de débat, la Maison-Blanche...",https://www.lesechos.fr/tech-medias/intelligen...
9,"En Europe, l'IA propulse surtout la « vieille ...",Le thème de l'intelligence artificielle a cont...,https://www.lesechos.fr/finance-marches/marche...


# Étape 9 - Exporter les résultats

## Enregistrement au format CSV

Le fichier est encodé en UTF-8 avec BOM afin de conserver correctement les accents lors de son ouverture dans Excel.

In [39]:
fichier_sortie = Path("articles_les_echos_ia.csv")
df_articles.to_csv(fichier_sortie, index=False, encoding="utf-8-sig")
print("Résultats enregistrés dans :", fichier_sortie.resolve())

Résultats enregistrés dans : C:\TCHIKSON\Webscrapping\articles_les_echos_ia.csv


# Étape 10 - Fermer le navigateur


In [40]:
driver.quit()
print("Navigateur fermé.")

Navigateur fermé.
